In [0]:
from pyspark.sql.functions import col, when, row_number, current_timestamp
from pyspark.sql.window import Window

SOURCE_TABLE = "banking.banking.banking_bronze_transactions"
TARGET_TABLE = "banking.banking.banking_silver_transactions"

print("🔹 Starting Enterprise Silver Layer")

In [0]:
if spark.catalog.tableExists(TARGET_TABLE):
    print("Incremental load detected")

    max_time = spark.sql(f"""
        SELECT MAX(ingestion_time) as max_time 
        FROM {TARGET_TABLE}
    """).collect()[0]["max_time"]

    df = spark.table(SOURCE_TABLE).filter(col("ingestion_time") > max_time)

else:
    print("First run - full load")
    df = spark.table(SOURCE_TABLE)

print(f"Records to process: {df.count()}")

In [0]:
# Remove bad records
df = df.filter(col("txn_id").isNotNull())
df = df.filter(col("amount") > 0)

# Track rejected records (optional audit)
invalid_count = spark.table(SOURCE_TABLE).count() - df.count()
print(f"Invalid records filtered: {invalid_count}")

In [0]:
df = df.withColumn(
    "transaction_type",
    when(col("transaction_type").isin("DEBIT", "CREDIT"), col("transaction_type"))
    .otherwise("UNKNOWN")
)

In [0]:
window_spec = Window.partitionBy("txn_id").orderBy(col("ingestion_time").desc())

df = df.withColumn("row_num", row_number().over(window_spec)) \
       .filter(col("row_num") == 1) \
       .drop("row_num")

In [0]:
df = df.withColumn(
    "amount_category",
    when(col("amount") > 100000, "HIGH")
    .when(col("amount") > 50000, "MEDIUM")
    .otherwise("LOW")
)

In [0]:
df = df.withColumn("silver_processed_time", current_timestamp())

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(TARGET_TABLE):

    delta_table = DeltaTable.forName(spark, TARGET_TABLE)

    delta_table.alias("target").merge(
        df.alias("source"),
        "target.txn_id = source.txn_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print("Upsert (MERGE) completed")

else:
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(TARGET_TABLE)

    print("Silver table created")

In [0]:
print("🔹 Data Quality Summary")

print(f"Total processed: {df.count()}")

df.groupBy("amount_category").count().show()

df.groupBy("transaction_type").count().show()